In [6]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [51]:
df=pd.read_csv('df_evaluated_30.csv')

In [52]:
import pandas as pd
import ast

def parse_list(x):
    """Convert a stringified Python list back into a list."""
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()
        try:
            return ast.literal_eval(x)
        except Exception:
            return []

    return []


rows = []

for _, row in df.iterrows():

    comments = parse_list(row["generated_comment"])
    systems = parse_list(row["generation_systems"])

    # Skip malformed rows
    if len(comments) != len(systems):
        print(
            f"Mismatch in patch {row['patch_id']}: "
            f"{len(comments)} comments vs {len(systems)} system groups"
        )
        continue

    for comment, sys_group in zip(comments, systems):

        # sys_group may still be a string like:
        # "['sys1', 'sys2', 'sys3']"
        if isinstance(sys_group, str):
            try:
                sys_group = ast.literal_eval(sys_group)
            except Exception:
                sys_group = [sys_group]

        # If it's just a single value, wrap it
        if not isinstance(sys_group, list):
            sys_group = [sys_group]

        for sys in sys_group:
            new_row = row.copy()

            new_row["generated_comment"] = comment
            new_row["generation_system"] = sys
            new_row["generated_by_multiple"] = len(sys_group) > 1
            new_row["all_generation_systems"] = sys_group

            rows.append(new_row)

df_exploded = pd.DataFrame(rows)


In [54]:
df_exploded.head()
# df=df_exploded.copy()

,hunk,human_comment,pr_title,changed_files,target_file,pr_url,github_commit_url,patch_id,generated_context,cluster_id,generated_comment,num_comments,generation_systems,categories,severities,Rel_Pred,per_comment_scores,per_comment_reasons,num_full_matches,num_partial_matches,best_score,best_comment_index,best_comment,match,generation_system,generated_by_multiple,all_generation_systems
0,"@@ -274,6 +274,7 @@ class RootPathHandler(BaseTaskHistoryHandler):\n self.redirect(""/static/visualiser/index.html"")\n \n def head(self):\n+ """"""HEAD endpoint for health checking the scheduler""""""\n self.set_status(204)\n self.finish()\n \n","Is the name ""head"" a convention for health checking? Regardless it caught me by surprise, maybe add some docs to this function on why it exist? It should also say what 204.",Add HEAD endpoint to scheduler server for status/health checks,"['luigi/server.py', 'test/server_test.py']",luigi/server.py,https://github.com/spotify/luigi/pull/2789,https://github.com/spotify/luigi/commit/03f712cffab169e7b617425c22eb84d82c8f081c,P000000,"class BaseTaskHistoryHandler(tornado.web.RequestHandler):\n def initialize(self, scheduler):\n self._scheduler = scheduler\n\n def get_template_path(self):\n return pkg_resources.resource_filename(__name__, 'templates')\n\n\nclass RootPathHandler(BaseTaskHistoryHandler):\n def get(self):\n self.redirect(""/static/visualiser/index.html"")\n\n def head(self):\n self.set_status(204)\n self.finish()","['P000000_c0', 'P000000_c1', 'P000000_c2']",The docstring claims this is a 'HEAD endpoint for health checking the scheduler' but the implementation only returns 204 without actually verifying the scheduler's health. Should this method check scheduler status or should the docstring be more accurate about what it does?,"[3, 3, 3]","[""['poolside/laguna-m.1:free']"", ""['cohere/north-mini-code:free']"", ""['nvidia/nemotron-3-ultra-550b-a55b:free']""]","[""['Documentation']"", ""['Documentation']"", ""['Documentation']""]","[""['Medium']"", ""['Low']"", ""['Low']""]","[5.0, 5.0, 5.0]","[1, 0, 2]","['Partial match – addresses docstring but suggests implementation change rather than just documentation.', 'No match – focuses on formatting, not documentation content.', 'Full match – directly addresses adding documentation for purpose and 204 meaning.']",1.0,1.0,2,2.0,"The docstring could be more descriptive — consider mentioning the 204 No Content response and that it returns an empty body, which is the standard for HEAD health checks.",True,poolside/laguna-m.1:free,False,[poolside/laguna-m.1:free]
0,"@@ -274,6 +274,7 @@ class RootPathHandler(BaseTaskHistoryHandler):\n self.redirect(""/static/visualiser/index.html"")\n \n def head(self):\n+ """"""HEAD endpoint for health checking the scheduler""""""\n self.set_status(204)\n self.finish()\n \n","Is the name ""head"" a convention for health checking? Regardless it caught me by surprise, maybe add some docs to this function on why it exist? It should also say what 204.",Add HEAD endpoint to scheduler server for status/health checks,"['luigi/server.py', 'test/server_test.py']",luigi/server.py,https://github.com/spotify/luigi/pull/2789,https://github.com/spotify/luigi/commit/03f712cffab169e7b617425c22eb84d82c8f081c,P000000,"class BaseTaskHistoryHandler(tornado.web.RequestHandler):\n def initialize(self, scheduler):\n self._scheduler = scheduler\n\n def get_template_path(self):\n return pkg_resources.resource_filename(__name__, 'templates')\n\n\nclass RootPathHandler(BaseTaskHistoryHandler):\n def get(self):\n self.redirect(""/static/visualiser/index.html"")\n\n def head(self):\n self.set_status(204)\n self.finish()","['P000000_c0', 'P000000_c1', 'P000000_c2']",The docstring is added but there is no blank line after it. Check if other docstrings in the file have a blank line and adjust accordingly for consistency.,"[3, 3, 3]","[""['poolside/laguna-m.1:free']"", ""['cohere/north-mini-code:free']"", ""['nvidia/nemotron-3-ultra-550b-a55b:free']""]","

In [55]:
import pandas as pd
import ast

def parse_list(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()
        try:
            return ast.literal_eval(x)
        except Exception:
            return []

    return []

rows = []

for _, row in df.iterrows():

    comments = parse_list(row["generated_comment"])
    systems = parse_list(row["generation_systems"])
    scores = parse_list(row["per_comment_scores"])

    # Make sure every comment has a system group and a score
    if not (len(comments) == len(systems) == len(scores)):
        print(
            f"Mismatch in patch {row['patch_id']}: "
            f"{len(comments)} comments, "
            f"{len(systems)} system groups, "
            f"{len(scores)} scores"
        )
        continue

    for comment, sys_group, score in zip(comments, systems, scores):

        # Convert "['sys1', 'sys2']" -> ['sys1', 'sys2']
        if isinstance(sys_group, str):
            try:
                sys_group = ast.literal_eval(sys_group)
            except Exception:
                sys_group = [sys_group]

        if not isinstance(sys_group, list):
            sys_group = [sys_group]

        # Duplicate the row once per generation system
        for sys in sys_group:
            new_row = row.copy()

            new_row["generated_comment"] = comment
            new_row["generation_system"] = sys
            new_row["per_comment_score"] = score
            new_row["generated_by_multiple"] = len(sys_group) > 1
            new_row["all_generation_systems"] = sys_group

            rows.append(new_row)

df_exploded = pd.DataFrame(rows)

In [58]:
df_exploded.info()

<class 'pandas.core.frame.DataFrame'>
Index: 114 entries, 0 to 28
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   hunk                    114 non-null    object 
 1   human_comment           114 non-null    object 
 2   pr_title                114 non-null    object 
 3   changed_files           114 non-null    object 
 4   target_file             114 non-null    object 
 5   pr_url                  90 non-null     object 
 6   github_commit_url       114 non-null    object 
 7   patch_id                114 non-null    object 
 8   generated_context       114 non-null    object 
 9   cluster_id              114 non-null    object 
 10  generated_comment       114 non-null    object 
 11  num_comments            114 non-null    object 
 12  generation_systems      114 non-null    object 
 13  categories              114 non-null    object 
 14  severities              114 non-null    object 


In [60]:
df=df_exploded.copy()

In [43]:
match_rate = (
    df.groupby("generation_system")
      .agg(
          total_comments=("match", "size"),
          matched=("match", "sum"),
          match_rate=("match", "mean"),
          avg_best_score=("best_score", "mean")
      )
)

match_rate["match_rate"] *= 100
match_rate = match_rate.sort_values("match_rate", ascending=False)

print(match_rate)

                                        total_comments  matched  match_rate  \
generation_system                                                             
nvidia/nemotron-3-super-120b-a12b:free              14        9   64.285714   
cohere/north-mini-code:free                         33       18   54.545455   
nvidia/nemotron-3-ultra-550b-a55b:free              22       11   50.000000   
tencent/hy3:free                                    24       11   45.833333   
poolside/laguna-m.1:free                            21        8   38.095238   

                                        avg_best_score  
generation_system                                       
nvidia/nemotron-3-super-120b-a12b:free        1.428571  
cohere/north-mini-code:free                   1.181818  
nvidia/nemotron-3-ultra-550b-a55b:free        1.181818  
tencent/hy3:free                              1.125000  
poolside/laguna-m.1:free                      1.000000  


In [61]:
summary = (
    df.groupby("generation_system")
      .agg(
          total_comments=("generated_comment", "count"),
          matched=("match", "sum"),
          match_rate=("match", "mean"),
          avg_score=("per_comment_score", "mean"),
          median_score=("per_comment_score", "median"),
      )
      .sort_values("match_rate", ascending=False)
)

summary["match_rate"] *= 100
summary

,total_comments,matched,match_rate,avg_score,median_score
generation_system,,,,,
nvidia/nemotron-3-super-120b-a12b:free,14,9,64.285714,1.142857,1.5
cohere/north-mini-code:free,33,18,54.545455,0.787879,0.0
nvidia/nemotron-3-ultra-550b-a55b:free,22,11,50.000000,0.909091,1.0
tencent/hy3:free,24,11,45.833333,0.791667,0.0
poolside/laguna-m.1:free,21,8,38.095238,0.761905,1.0
